In [1]:
import random
from abc import ABC, abstractmethod

class ArithmeticGenerator(ABC):
    """算术表达式生成器基类"""
    @abstractmethod
    def generate(self, max_terms=3, max_digits=2, min_val=0, max_val=100):
        pass


class ModNArithmeticGenerator(ArithmeticGenerator):
    """
    模 N 算术表达式生成器，支持括号嵌套
    """
    
    def __init__(self, n=10, allow_parentheses=True, max_depth=2):
        """
        Args:
            n: 模数
            allow_parentheses: 是否允许生成括号
            max_depth: 最大括号嵌套深度
        """
        self.n = n
        self.allow_parentheses = allow_parentheses
        self.max_depth = max_depth
        self._operators = ['+', '-']
    
    def _generate_expression(self, depth=0):
        """
        递归生成表达式，支持括号嵌套
        """
        if depth < self.max_depth and random.random() < 0.3:
            # 生成带括号的子表达式
            left = self._generate_expression(depth + 1)
            op = random.choice(self._operators)
            right = self._generate_expression(depth + 1)
            return f"({left} {op} {right})"
        else:
            # 生成叶子节点（数字）
            return str(random.randint(0, self.n - 1))
    
    def _generate_sequence(self, max_terms, depth=0):
        """
        生成一个包含多个 term 的序列，可能包含括号
        """
        if max_terms == 0:
            return self._generate_expression(depth)
        
        if max_terms == 1:
            return self._generate_expression(depth)
        
        # 随机决定是否在当前层级分组
        if depth < self.max_depth and random.random() < 0.2:
            # 将多个 term 用括号包裹
            num_terms = random.randint(2, max_terms)
            parts = []
            for i in range(num_terms):
                parts.append(self._generate_expression(depth + 1))
                if i < num_terms - 1:
                    parts.append(random.choice(self._operators))
            expr = ' '.join(parts)
            return f"({expr})"
        else:
            # 直接拼接
            num_terms = random.randint(2, max_terms)
            parts = []
            for i in range(num_terms):
                parts.append(self._generate_expression(depth))
                if i < num_terms - 1:
                    parts.append(random.choice(self._operators))
            return ' '.join(parts)
    
    def generate(self, max_terms=3, max_digits=2, min_val=0, max_val=100):
        """
        生成一个模 N 算术表达式
        
        Returns:
            (expr, result): 表达式字符串和结果字符串
        """
        # 生成表达式字符串
        expr = self._generate_sequence(max_terms)
        
        # 安全计算表达式值（支持括号）
        try:
            # 用 eval 计算，但只允许数字和运算符
            # 注意：这里为了安全，只允许数字、运算符、括号和空格
            allowed_chars = set('0123456789+-*/() ')
            if not all(c in allowed_chars for c in expr):
                raise ValueError("Expression contains invalid characters")
            total = eval(expr)
            # 确保结果是整数（避免浮点数）
            total = int(total)
        except Exception as e:
            # 如果计算失败，回退到简单模式
            print(f"Warning: eval failed for {expr}, falling back to simple mode")
            return self._generate_simple(max_terms, min_val, max_val)
        
        # 应用模运算
        result = total % self.n
        return expr + ' = ', str(result)
    
    def _generate_simple(self, max_terms, min_val, max_val):
        """简单模式（无括号）的回退方案"""
        num_terms = random.randint(2, max_terms)
        tokens = []
        total = 0
        for i in range(num_terms):
            num = random.randint(0, self.n - 1)
            if i == 0:
                total = num
            else:
                op = random.choice(self._operators)
                if op == '+':
                    total += num
                else:
                    total -= num
                tokens.append(op)
            tokens.append(str(num))
        result = total % self.n
        expr = ' '.join(tokens)
        return expr + ' = ', str(result)
    
    def build_vocab(self):
        """构建模 N 运算的词表"""
        digits = [str(i) for i in range(self.n)]
        return digits + ['+', '-', '=', ' ', '(', ')', '<SOS>', '<EOS>', '<PAD>']
    
    def __call__(self, *args, **kwargs):
        return self.generate(*args, **kwargs)


# 使用示例
if __name__ == "__main__":
    # 模 3 运算，支持括号
    gen = ModNArithmeticGenerator(n=3, allow_parentheses=True, max_depth=2)
    
    for i in range(20):
        expr, result = gen.generate(max_terms=4)
        print(f"{expr}{result} (mod 3)")

0 - 2 + 1 = 2 (mod 3)
(0 + 1) - 1 = 0 (mod 3)
((2 - 2) + 1 + 0 + 2) = 0 (mod 3)
((2 - 0) - 1 - 1) = 0 (mod 3)
1 - ((1 - 1) + 1) + 2 + (1 + (0 - 1)) = 2 (mod 3)
(2 + (2 - 0)) + 1 + (0 - 2) - (1 + 0) = 2 (mod 3)
(0 + (1 + 0)) - 0 - ((2 + 2) - (0 - 1)) + 1 = 0 (mod 3)
(0 - (0 + 0) + 1) = 1 (mod 3)
1 + 1 + (0 - (2 + 0)) - ((1 + 2) + (1 - 1)) = 0 (mod 3)
1 - 0 + (0 + 2) = 0 (mod 3)
1 - (2 + 2) - 0 = 0 (mod 3)
2 + 2 + 2 = 0 (mod 3)
0 + 0 + 0 = 0 (mod 3)
(0 + 0) - 1 + 0 = 2 (mod 3)
2 - (2 - (1 + 0)) = 1 (mod 3)
((1 + 2) + 2) - 1 - (0 - 0) = 1 (mod 3)
(1 + 1) - ((1 + 2) + 0) - 2 + (2 + (1 - 1)) = 2 (mod 3)
0 - 0 = 0 (mod 3)
0 - (1 + 1) + 1 = 2 (mod 3)
1 - 2 = 2 (mod 3)


In [14]:
import random
from abc import ABC, abstractmethod

class ArithmeticGenerator(ABC):
    @abstractmethod
    def generate(self, max_terms=3, max_digits=2, min_val=0, max_val=100):
        pass


class ModNArithmeticGenerator(ArithmeticGenerator):
    """
    模 N 算术表达式生成器，支持括号嵌套，mod n 放在等号左侧
    """
    
    def __init__(self, n=None, allow_parentheses=True, max_depth=2):
        self.fixed_n = n
        self.allow_parentheses = allow_parentheses
        self.max_depth = max_depth
        self._operators = ['+', '-']
        self._mod_range = (2, 10)
    
    def _get_mod_value(self):
        if self.fixed_n is not None:
            return self.fixed_n
        return random.randint(self._mod_range[0], self._mod_range[1])
    
    def _generate_expression(self, depth=0, n=10, wrap_outer=False):
        """递归生成表达式，支持括号嵌套"""
        if depth < self.max_depth and random.random() < 0.3:
            left = self._generate_expression(depth + 1, n)
            op = random.choice(self._operators)
            right = self._generate_expression(depth + 1, n)
            expr = f"({left} {op} {right})"
        else:
            expr = str(random.randint(0, n - 1))
        
        # 如果外层需要括号且当前表达式不是单独的叶子节点，可以加括号
        if wrap_outer and random.random() < 0.4 and depth == 0:
            return f"({expr})"
        return expr
    
    def _generate_sequence(self, max_terms, depth=0, n=10):
        """生成包含多个 term 的序列"""
        if max_terms == 0:
            return self._generate_expression(depth, n)
        if max_terms == 1:
            return self._generate_expression(depth, n)
        
        if depth < self.max_depth and random.random() < 0.2:
            num_terms = random.randint(2, max_terms)
            parts = []
            for i in range(num_terms):
                parts.append(self._generate_expression(depth + 1, n))
                if i < num_terms - 1:
                    parts.append(random.choice(self._operators))
            expr = ' '.join(parts)
            return f"({expr})"
        else:
            num_terms = random.randint(2, max_terms)
            parts = []
            for i in range(num_terms):
                parts.append(self._generate_expression(depth, n))
                if i < num_terms - 1:
                    parts.append(random.choice(self._operators))
            return ' '.join(parts)
    
    def generate(self, simple=True,max_terms=3, max_digits=2, min_val=0, max_val=100):
        """生成模 N 算术表达式"""
        n = self._get_mod_value()
        if simple==True:
            return self._generate_simple(max_terms,n)
        else:
            # 生成表达式
            expr = self._generate_sequence(max_terms, 0, random.randint(2,n))

            # ★ 随机决定是否给最外层加括号（可选）
            if random.random() < 0.3:
                expr = f"({expr})"

            # 安全计算表达式值
            try:
                allowed_chars = set('0123456789+-*/() ')
                if not all(c in allowed_chars for c in expr):
                    raise ValueError("Expression contains invalid characters")
                total = int(eval(expr))
            except Exception:
                return self._generate_simple(max_terms, n)

            result = total % n
            return f"({expr}) mod {n} = ", str(result), n
    
    def _generate_simple(self, max_terms, n):
        """简单模式（无括号）的回退方案"""
        num_terms = random.randint(2, max_terms)
        tokens = []
        total = 0
        for i in range(num_terms):
            num = random.randint(0, n - 1)
            if i == 0:
                total = num
            else:
                op = random.choice(self._operators)
                if op == '+':
                    total += num
                else:
                    total -= num
                tokens.append(op)
            tokens.append(str(num))
        result = total % n
        expr = ' '.join(tokens)
        if random.random() < 0.3:
            expr = f"({expr})"
        return f"({expr}) mod {n} = ", str(result), n
    
    def build_vocab(self):
        """★ 固定词表：0-9 全部包含，不依赖模数"""
        digits = [str(i) for i in range(10)]
        return digits + ['+', '-', '=', ' ', '(', ')', '<SOS>', '<EOS>', '<PAD>', 'mod']
    
    def __call__(self, *args, **kwargs):
        return self.generate(*args, **kwargs)


# 使用示例
if __name__ == "__main__":
    gen = ModNArithmeticGenerator(n=None)
    for i in range(10):
        expr, result, n = gen.generate(simple=False,max_terms=4)
        print(f"{expr}{result} (mod {n})")
    print("---------------------------------")
    for i in range(10):
        expr, result, n = gen.generate(simple=True,max_terms=4)
        print(f"{expr}{result} (mod {n})")

(((1 + 1) - 2 + (2 + 1))) mod 10 = 3 (mod 10)
(((1 + 3) + 4)) mod 10 = 8 (mod 10)
((0 + 4) + 5) mod 10 = 9 (mod 10)
(4 - (4 - 3)) mod 10 = 3 (mod 10)
(2 + ((2 - 2) + 0)) mod 10 = 2 (mod 10)
((0 - 4) + 2 - 0 - (2 - 0)) mod 10 = 6 (mod 10)
((4 - 2 - (2 - 0))) mod 10 = 0 (mod 10)
(((5 + 2) + 4)) mod 10 = 1 (mod 10)
((3 + (1 + 0)) - 3 + 2) mod 10 = 3 (mod 10)
(3 + (0 - 0) - 5 - 2) mod 10 = 6 (mod 10)
---------------------------------
(5 - 9 - 4) mod 10 = 2 (mod 10)
((8 - 4 + 1)) mod 10 = 5 (mod 10)
(5 + 6 + 6 + 3) mod 10 = 0 (mod 10)
(2 - 2 + 7 + 8) mod 10 = 5 (mod 10)
(1 - 8) mod 10 = 3 (mod 10)
(2 + 2 - 0 + 5) mod 10 = 9 (mod 10)
(8 - 8 + 9) mod 10 = 9 (mod 10)
(0 - 9) mod 10 = 1 (mod 10)
(8 - 4) mod 10 = 4 (mod 10)
(2 - 5) mod 10 = 7 (mod 10)
